# P1. Train an LLM From Scratch (Capstone)

**Tier:** Projects
**Estimated time:** 75 minutes
**Prerequisites:** 01, 02, 04, 05, 07
**Priority:** 🟢 Nice-to-have — deep-understanding and portfolio value, but AI Engineers rarely train a model from scratch on the job; Tiers 1-2 already cover the underlying concepts. *If skipped, revisit when:* before interviews that probe fundamentals, or when debugging a fine-tuning/training job where knowing what's happening under the hood actually matters.
**Source material:** Fareed Khan's "Train LLM From Scratch" — MIT license — https://github.com/FareedKhan-dev/train-llm-from-scratch (full pipeline: Data → Pretraining → SFT → Reward Model → DPO → PPO → GRPO, hand-written PyTorch, scales to a 2B-parameter model on a single GPU using The Pile)

## What You'll Learn
- Building every stage of a decoder-only transformer from scratch: tokenizer, embeddings, multi-head causal self-attention, transformer block, full model
- A real training loop (not a toy `.fit()` call) that visibly reduces loss on a real corpus
- Autoregressive text generation with temperature sampling
- How this ~13M-parameter, CPU/MPS-scale build maps onto Fareed Khan's repo structure for scaling to a 2B-parameter model on a single GPU

## Why This Matters
Every concept from Tier 1 (attention, transformers) and Tier 2 (pretraining, scaling) has been explained and demonstrated in isolation. This capstone assembles all of them into one working model you train end-to-end, so "attention is a weighted average over value vectors" stops being a sentence you can recite and becomes code you watched actually learn something. It won't produce a capable model — 13M parameters and a few hundred steps can't — but the *pipeline* is structurally the same one that scales to billions of parameters, which is the whole point.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(0)

def pick_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = pick_device()
print(f"Device: {device}  (GPU-check: use this device for anything below — code is device-agnostic)")


## Stage 1 — Data

Fareed Khan's repo trains on The Pile, a large real-world corpus, on a real GPU. We use a small synthetic corpus (a handful of proverbs and idioms, repeated with variation) so the ENTIRE pipeline — tokenizer through generation — runs in well under a minute on a laptop CPU/MPS, while exercising the exact same code shape a much larger run would use.

In [ ]:
PHRASES = [
    "the early bird catches the worm",
    "actions speak louder than words",
    "practice makes perfect every time",
    "a stitch in time saves nine",
    "the pen is mightier than the sword",
    "honesty is always the best policy",
    "slow and steady wins the race",
    "knowledge is power when applied",
]
# Repeat with light shuffling so the model sees varied ORDER but the same underlying phrases —
# enough structure for a tiny model to visibly learn local patterns within a short training run.
rng = np.random.default_rng(0)
lines = []
for _ in range(300):
    p = PHRASES[rng.integers(len(PHRASES))]
    lines.append(p)
text = " . ".join(lines) + " ."
print(f"Corpus length: {len(text):,} characters")
print(f"Sample: {text[:120]!r}")


## Stage 2 — Tokenizer (character-level, from scratch)

The simplest possible tokenizer: the vocabulary IS the set of unique characters in the corpus, and encoding/decoding is a dictionary lookup. This is deliberately the least sophisticated choice (compare to notebook 02's BPE) — it keeps every other stage's code focused on the transformer itself rather than tokenization edge cases.

In [ ]:
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

def encode(s):
    return [stoi[c] for c in s]

def decode(ids):
    return "".join(itos[i] for i in ids)

print(f"Vocab size: {vocab_size} unique characters: {''.join(chars)!r}")
data = torch.tensor(encode(text), dtype=torch.long)
print(f"Encoded corpus: {data.shape[0]:,} tokens")
print(f"Round-trip check: {decode(encode('the early bird')) == 'the early bird'}")


## Stage 3 — Embeddings

Every token id becomes a learned vector (token embedding), and every position in the sequence gets its own learned vector too (positional embedding, notebook 5's approach — a learned table here rather than the sinusoidal encoding used there, to show the alternative). The two are added together, exactly as covered conceptually in notebook 3 (embeddings) and notebook 5 (transformer architecture).

In [ ]:
D_MODEL = 384
BLOCK_SIZE = 192   # max context length this model can attend over

token_embedding = nn.Embedding(vocab_size, D_MODEL).to(device)
position_embedding = nn.Embedding(BLOCK_SIZE, D_MODEL).to(device)

sample_batch = data[:BLOCK_SIZE].unsqueeze(0).to(device)   # shape (1, BLOCK_SIZE)
tok_emb = token_embedding(sample_batch)
pos_emb = position_embedding(torch.arange(BLOCK_SIZE, device=device))
combined = tok_emb + pos_emb
print(f"Token embedding shape:    {tuple(tok_emb.shape)}")
print(f"Position embedding shape: {tuple(pos_emb.shape)}")
print(f"Combined (token + position) shape: {tuple(combined.shape)}")


## Stage 4 — Multi-head causal self-attention, from scratch

This is the same scaled dot-product attention from notebook 4, extended to multiple heads (notebook 4's Exercise 3) and made CAUSAL — a mask that prevents position `t` from attending to any position after it, which is what makes this a valid next-token predictor instead of a bidirectional encoder like BERT (notebook 5's Lecture 2 material).

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, block_size):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.qkv_proj = nn.Linear(d_model, 3 * d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        # A lower-triangular mask: position i can only attend to positions <= i.
        causal_mask = torch.tril(torch.ones(block_size, block_size))
        self.register_buffer("causal_mask", causal_mask.view(1, 1, block_size, block_size))

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv_proj(x)
        q, k, v = qkv.split(C, dim=2)
        # Reshape (B, T, C) -> (B, n_heads, T, d_head) so each head attends independently.
        q = q.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_head).transpose(1, 2)

        attn_scores = (q @ k.transpose(-2, -1)) / (self.d_head ** 0.5)
        attn_scores = attn_scores.masked_fill(self.causal_mask[:, :, :T, :T] == 0, float("-inf"))
        attn_weights = F.softmax(attn_scores, dim=-1)
        out = attn_weights @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.out_proj(out)

# Sanity check: causal masking means output at position 0 must not depend on later positions.
test_attn = CausalSelfAttention(D_MODEL, n_heads=6, block_size=BLOCK_SIZE).to(device)
x_test = torch.randn(1, BLOCK_SIZE, D_MODEL, device=device)
out_a = test_attn(x_test)
x_test_modified = x_test.clone()
x_test_modified[:, -1, :] = torch.randn(D_MODEL, device=device)   # change only the LAST position
out_b = test_attn(x_test_modified)
print("Output at position 0 unaffected by changing the LAST position:",
      torch.allclose(out_a[:, 0, :], out_b[:, 0, :], atol=1e-6))


## Stage 5 — Transformer block and the full model

A block is attention + a feed-forward network, each wrapped in a residual connection and pre-normalized with LayerNorm (notebook 5's architecture). Stacking several blocks, then a final projection back to vocabulary size, gives the complete decoder-only model — the same shape as Fareed Khan's repo's core transformer, at a much smaller scale.

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, block_size, ff_mult=4):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, block_size)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, ff_mult * d_model),
            nn.GELU(),
            nn.Linear(ff_mult * d_model, d_model),
        )

    def forward(self, x):
        x = x + self.attn(self.ln1(x))   # residual around attention
        x = x + self.mlp(self.ln2(x))    # residual around the feed-forward network
        return x

class TinyGPT(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, n_layers, block_size):
        super().__init__()
        self.block_size = block_size
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(block_size, d_model)
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, block_size) for _ in range(n_layers)
        ])
        self.ln_final = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

    def forward_hidden(self, idx):
        """Returns final hidden states before the vocab projection — every downstream head
        (here just next-token prediction) composes on top of this, matching the design Fareed
        Khan's repo uses so SFT/DPO/RL heads can all reuse the same base model unmodified."""
        B, T = idx.shape
        positions = torch.arange(T, device=idx.device)
        x = self.token_embedding(idx) + self.position_embedding(positions)
        for block in self.blocks:
            x = block(x)
        return self.ln_final(x)

    def forward(self, idx):
        hidden = self.forward_hidden(idx)
        return self.head(hidden)

model = TinyGPT(vocab_size, d_model=D_MODEL, n_heads=6, n_layers=8, block_size=BLOCK_SIZE).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,} ({n_params / 1e6:.1f}M)")


## Stage 6 — Training loop

Sample random windows of `BLOCK_SIZE` characters, predict the NEXT character at every position (shifted-by-one targets, the standard next-token objective from notebook 6), and optimize cross-entropy loss with AdamW. This is a genuine training loop — the same one a 2B-parameter run uses, just far fewer steps and far less data.

In [ ]:
def get_batch(batch_size, block_size):
    max_start = len(data) - block_size - 1
    starts = torch.randint(0, max_start, (batch_size,))
    x = torch.stack([data[s:s + block_size] for s in starts]).to(device)
    y = torch.stack([data[s + 1:s + block_size + 1] for s in starts]).to(device)
    return x, y

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
BATCH_SIZE, N_STEPS = 32, 300
losses = []

model.train()
for step in range(N_STEPS):
    xb, yb = get_batch(BATCH_SIZE, BLOCK_SIZE)
    logits = model(xb)
    loss = F.cross_entropy(logits.view(-1, vocab_size), yb.view(-1))
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    if step % 50 == 0 or step == N_STEPS - 1:
        print(f"step {step:4d}   loss {loss.item():.3f}")


In [ ]:
%matplotlib inline
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(losses)
ax.set_xlabel("training step")
ax.set_ylabel("cross-entropy loss")
ax.set_title(f"Training loss over {N_STEPS} steps ({n_params/1e6:.1f}M-parameter model)")
plt.tight_layout()
plt.show()


*Loss should fall from roughly `log(vocab_size)` (a random, untrained model) toward a much lower value as the model learns the corpus's repeated phrase structure — at this tiny scale and step count, expect visible memorization of local patterns, not real language understanding (the scaling-laws point from notebook 7: capability emerges from scale, which 13M parameters and 300 steps simply doesn't have).*

## Stage 7 — Generation

Autoregressive sampling: feed the model's own output back in as the next input, one character at a time, using temperature to control how deterministic vs. varied the sampling is (notebook 6's temperature concept). Compare a prompt completion BEFORE vs. AFTER training to see what the model actually learned.

In [ ]:
@torch.no_grad()
def generate(model, prompt, n_new_tokens=80, temperature=0.8):
    model.eval()
    idx = torch.tensor([encode(prompt)], device=device)
    for _ in range(n_new_tokens):
        idx_cond = idx[:, -BLOCK_SIZE:]
        logits = model(idx_cond)[:, -1, :] / temperature
        probs = F.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_id], dim=1)
    model.train()
    return decode(idx[0].tolist())

prompt = "the early"
print("AFTER training:")
print(" ", generate(model, prompt))

# For comparison: a freshly initialized (untrained) model of the identical architecture.
untrained_model = TinyGPT(vocab_size, d_model=D_MODEL, n_heads=6, n_layers=8, block_size=BLOCK_SIZE).to(device)
print("\nUNTRAINED (random weights, same architecture):")
print(" ", generate(untrained_model, prompt))


## Scaling to a 2B-parameter model on a single GPU

Every stage above — tokenizer, embeddings, causal attention, transformer block, training loop, generation — is structurally IDENTICAL to what Fareed Khan's repo does at 2B parameters; only the numbers change:

| | This notebook | Fareed Khan's repo (2B config) |
|---|---|---|
| Tokenizer | character-level (100 tokens) | BPE, ~32-50k tokens |
| `d_model` | 384 | ~2048+ |
| `n_layers` | 8 | ~24+ |
| Corpus | 300 repeated proverbs (~15k chars) | The Pile (hundreds of billions of tokens) |
| Training steps | 300 | hundreds of thousands, with learning-rate schedules and gradient accumulation |
| Hardware | CPU/MPS, ~30 seconds | a single A100-class GPU, days |
| Post-training | none | SFT → Reward Model → DPO → PPO → GRPO (notebooks 8/9 cover these concepts) |

The `forward_hidden` method above exists for exactly this reason: it's the seam Fareed Khan's repo builds every post-training head (SFT loss, reward model, DPO/PPO log-probs) on top of, without touching the base transformer's code. If you want to go further than this notebook, the repo's own docs walk through each of those stages using this exact same base-model shape.

## Exercises

**Exercise 1 (Warm-up):** Change `N_STEPS` to 600 and re-plot the loss curve. Does the loss keep dropping, plateau, or does generation quality visibly improve further?

**Exercise 2 (Apply):** Implement `count_parameters_by_component(model)` that reports how many parameters live in embeddings vs. attention vs. feed-forward vs. the output head — confirm the feed-forward layers (the `4x` expansion) dominate, as the "12·d_model²-per-layer" napkin math predicts.

**Exercise 3 (Extend):** This model has no positional encoding beyond `BLOCK_SIZE` — sketch (in comments) what would need to change to support RoPE (notebook 5's Lecture 2 material) instead of a learned position embedding table, and why that would let the model generalize to sequences longer than it was trained on.


In [ ]:
# Exercise 1: Warm-up
# Task: Re-train with N_STEPS = 600 (reuse get_batch/model/optimizer) and re-plot the loss curve.
# Hint: re-run the training loop cell with N_STEPS changed; compare the new loss curve's tail.

# YOUR CODE HERE


# Exercise 2: Apply
# Task: Implement count_parameters_by_component(model) -> dict grouping by embeddings/attn/mlp/head.
# Hint: iterate model.named_parameters() and match on substrings like "embedding", "attn", "mlp".

# YOUR CODE HERE


# Exercise 3: Extend
# Task: Sketch what changes to swap the learned position_embedding for RoPE.
# Hint: RoPE rotates the Q/K vectors by a position-dependent angle INSIDE attention, rather than
# adding a learned vector to the input BEFORE attention — it never needs a fixed BLOCK_SIZE table.

# YOUR CODE HERE


<details>
<summary>Click to reveal solutions</summary>

```python
# Exercise 1
N_STEPS_LONG = 600
losses_long = []
model_long = TinyGPT(vocab_size, d_model=D_MODEL, n_heads=6, n_layers=8, block_size=BLOCK_SIZE).to(device)
opt_long = torch.optim.AdamW(model_long.parameters(), lr=3e-4)
for step in range(N_STEPS_LONG):
    xb, yb = get_batch(BATCH_SIZE, BLOCK_SIZE)
    logits = model_long(xb)
    loss = F.cross_entropy(logits.view(-1, vocab_size), yb.view(-1))
    opt_long.zero_grad(); loss.backward(); opt_long.step()
    losses_long.append(loss.item())
# Expect diminishing returns: loss keeps falling but the SLOPE flattens — this tiny corpus
# has limited structure left to learn after a few hundred steps.

# Exercise 2
def count_parameters_by_component(model):
    groups = {"embeddings": 0, "attention": 0, "mlp": 0, "head": 0, "other": 0}
    for name, p in model.named_parameters():
        if "embedding" in name:
            groups["embeddings"] += p.numel()
        elif "attn" in name:
            groups["attention"] += p.numel()
        elif "mlp" in name:
            groups["mlp"] += p.numel()
        elif name.startswith("head"):
            groups["head"] += p.numel()
        else:
            groups["other"] += p.numel()
    return groups

print(count_parameters_by_component(model))

# Exercise 3
# RoPE (Rotary Position Embedding) would replace `self.position_embedding` entirely:
#   1. Remove position_embedding and the `+ self.position_embedding(positions)` line.
#   2. Inside CausalSelfAttention.forward, AFTER computing q and k (before the attention score
#      matmul), apply a rotation to each (q, k) pair based on its sequence position — pairs of
#      dimensions are rotated by an angle proportional to position * frequency.
#   3. Because the rotation is applied inside attention using RELATIVE position math, the model
#      never needs a fixed-size lookup table — it can be evaluated at sequence lengths longer
#      than anything seen during training, which a learned nn.Embedding(block_size, d_model)
#      table structurally cannot do (notebook 5's Lecture 2 material covers this in depth).
```
</details>

## Key Takeaways
- Every abstract concept from Tier 1 (attention, transformers) and Tier 2 (pretraining) is, underneath, exactly this: a tokenizer, an embedding table, a stack of attention+MLP blocks, a training loop, and a sampling function.
- Multi-head CAUSAL self-attention is the same scaled dot-product attention from notebook 4, plus a triangular mask that prevents attending to future positions — the one addition that turns an encoder into a valid autoregressive generator.
- `forward_hidden()` — returning hidden states before the vocab projection — is the seam that lets SFT, reward-model, and RL heads (notebooks 8/9) all build on the same base model unmodified, exactly as Fareed Khan's repo structures it.
- At 13M parameters and 300 steps, expect memorization of local patterns, not language understanding — that gap IS the scaling-laws lesson from notebook 7, made concrete instead of abstract.
- Scaling this exact pipeline to 2B parameters changes hyperparameters and hardware, not the code's shape — tokenizer, attention, blocks, training loop, and generation all stay structurally identical.

## What's Next
P2 builds a production RAG pipeline, synthesizing Tier 3's building blocks with Tier 5's evaluation discipline — a different kind of system, evaluated the way this one wasn't.
